In [ ]:
# 1. Import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings("ignore")

# 2. Load dataset
train = pd.read_csv("C:/Users/ghwns/Desktop/Competition/scu_ai_competition 2025/Data/campaign_train.csv")
test = pd.read_csv("C:/Users/ghwns/Desktop/Competition/scu_ai_competition 2025/Data/campaign_test.csv")
target = train["target"]

# 3. Fill missing values
for col in ["고객_교육수준", "고객_결혼여부"]:
    train[col].fillna(train[col].mode()[0], inplace=True)
    test[col].fillna(train[col].mode()[0], inplace=True)
train["고객_소득"].fillna(train["고객_소득"].median(), inplace=True)
test["고객_소득"].fillna(train["고객_소득"].median(), inplace=True)

# 4. Feature Engineering
def feature_engineering(df):
    df["총_구매금액"] = (
        df["고객_와인_구매금액"] + df["고객_과일_구매금액"] + df["고객_육류_구매금액"] +
        df["고객_생선_구매금액"] + df["고객_사탕_구매금액"] + df["고객_골드_구매금액"]
    )
    df["고객_나이"] = 2025 - df["출생연도"]
    df["과거_캠페인_수락횟수"] = df[[f"캠페인{i}_수락여부" for i in range(1, 6)]].sum(axis=1)
    df["고객_캠페인_반응성"] = df["과거_캠페인_수락횟수"] / 5
    df["고객_구매_소득비"] = df["총_구매금액"] / (df["고객_소득"] + 1)
    df["고객_청소년_비율"] = df["고객_청소년수"] / (df["고객_자녀수"] + 1)
    df["고객_총_구매횟수"] = df[["고객_회사사이트_통한_구매횟수", "고객_카탈로그_통한_구매횟수", "고객_매장방문_구매횟수"]].sum(axis=1)
    df["고객_사이트_구매비율"] = df["고객_회사사이트_통한_구매횟수"] / (df["고객_총_구매횟수"] + 1)
    df["고객_최근방문_환산"] = df["고객_지난달_회사사이트_방문횟수"] / (df["고객_최신구매일_경과기간"] + 1)
    df["고객_사이트_총_활동"] = df["고객_회사사이트_통한_구매횟수"] + df["고객_지난달_회사사이트_방문횟수"]

    # Additional interaction and ratio features
    df["소득x총구매"] = df["고객_소득"] * df["총_구매금액"]
    df["소득x카탈로그구매"] = df["고객_소득"] * df["고객_카탈로그_통한_구매횟수"]
    df["캠페인x웹방문"] = df["과거_캠페인_수락횟수"] * df["고객_지난달_회사사이트_방문횟수"]
    df["와인x캠페인"] = df["고객_와인_구매금액"] * df["고객_캠페인_반응성"]
    df["와인_구매비율"] = df["고객_와인_구매금액"] / (df["총_구매금액"] + 1)
    df["웹방문_x_소득"] = df["고객_지난달_회사사이트_방문횟수"] * df["고객_소득"]
    df["캠페인_x_총구매"] = df["과거_캠페인_수락횟수"] * df["총_구매금액"]
    df["소득_차이"] = df["고객_소득"] - df["고객_구매_소득비"]
    df["청소년x와인"] = df["고객_청소년수"] * df["고객_와인_구매금액"]
    df["방문x캠페인성향"] = df["고객_최근방문_환산"] * df["고객_캠페인_반응성"]
    df["총구매_비용효율"] = df["총_구매금액"] / (df["고객_총_구매횟수"] + 1)
    df["소득_대비_나이"] = df["고객_소득"] / (df["고객_나이"] + 1)
    df["웹방문_최신구매비"] = df["고객_지난달_회사사이트_방문횟수"] / (df["고객_최신구매일_경과기간"] + 1)
    df["소득기반_총구매"] = df["총_구매금액"] / (df["고객_소득"] + 1) * 10
    return df

train = feature_engineering(train)
test = feature_engineering(test)

# 5. Clustering-based Features
def make_cluster_features(train, test, cols, n_clusters, prefix):
    all_data = pd.concat([train[cols], test[cols]], axis=0)
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    labels = kmeans.fit_predict(all_data)
    train[f"{prefix}_클러스터"] = labels[:len(train)]
    test[f"{prefix}_클러스터"] = labels[len(train):]
    return train, test

train, test = make_cluster_features(train, test,
    ["고객_최근방문_환산", "고객_캠페인_반응성", "고객_총_구매횟수"], 4, "행동패턴")
train, test = make_cluster_features(train, test,
    ["총_구매금액", "고객_카탈로그_통한_구매횟수", "고객_회사사이트_통한_구매횟수"], 4, "구매패턴")
train, test = make_cluster_features(train, test,
    ["고객_소득", "고객_구매_소득비", "소득기반_총구매"], 5, "소득패턴")
train, test = make_cluster_features(train, test,
    ["고객_최신구매일_경과기간", "고객_지난달_회사사이트_방문횟수", "고객_사이트_총_활동"], 5, "시간패턴")

# 6. Label Encoding
le = LabelEncoder()
for col in ["고객_교육수준", "고객_결혼여부"]:
    train[col] = le.fit_transform(train[col])
    test[col] = le.transform(test[col])

# 7. Standardization
drop_cols = ["ID", "target", "고객_가입날짜"]
features = [col for col in train.columns if col not in drop_cols]
scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(train[features]), columns=features)
X_test = pd.DataFrame(scaler.transform(test[features]), columns=features)

# 8. Randomized Search for LGBM
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators': [300, 400, 500, 600],
    'learning_rate': [0.01, 0.02, 0.03, 0.05],
    'max_depth': [3, 4, 5, 6],
    'num_leaves': [20, 31, 40, 50],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'reg_alpha': [0, 0.1, 0.5],
    'reg_lambda': [0, 0.1, 0.5],
}

lgbm_base = LGBMClassifier(random_state=42)
random_search = RandomizedSearchCV(
    estimator=lgbm_base,
    param_distributions=param_dist,
    n_iter=30,
    cv=3,
    scoring='roc_auc',
    verbose=1,
    n_jobs=-1,
    random_state=42
)
random_search.fit(X_train, target)
best_lgbm = random_search.best_estimator_
print("Best LGBM Params:", random_search.best_params_)

# 9. Final Model: VotingClassifier (6:2:2)
model_rf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42)
model_lr = LogisticRegression(penalty='l2', solver='liblinear', random_state=42)

voting_model = VotingClassifier(
    estimators=[('lgbm', best_lgbm), ('rf', model_rf), ('lr', model_lr)],
    voting='soft',
    weights=[6, 2, 2],
    n_jobs=-1
)

# 10. Cross-Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
auc_scores = []

for train_idx, val_idx in cv.split(X_train, target):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = target.iloc[train_idx], target.iloc[val_idx]

    voting_model.fit(X_tr, y_tr)
    val_pred = voting_model.predict_proba(X_val)[:, 1]
    auc_scores.append(roc_auc_score(y_val, val_pred))

print("[CV] VotingClassifier (6:2:2, derived features + clustering + LGBM tuned) AUC:", np.mean(auc_scores))

# 11. Final Prediction
voting_model.fit(X_train, target)
pred = voting_model.predict_proba(X_test)[:, 1]
submit = pd.read_csv("C:/Users/ghwns/Desktop/Competition/scu_ai_competition 2025/Submission/campaign_sample_submission.csv")
submit["target"] = pred
submit.to_csv("C:/Users/ghwns/Desktop/Competition/scu_ai_competition 2025/Submission/73.submission.csv", index=False)


Fitting 3 folds for each of 30 candidates, totalling 90 fits
[LightGBM] [Info] Number of positive: 200, number of negative: 1144
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002145 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5250
[LightGBM] [Info] Number of data points in the train set: 1344, number of used features: 50
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.148810 -> initscore=-1.743969
[LightGBM] [Info] Start training from score -1.743969
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f